# Session 3: ChromaDB — Vector Storage and Retrieval

**Goal:** Move from in-memory search to a real vector database with persistence, metadata, and filtering.

| What you'll learn | Why it matters |
|---|---|
| ChromaDB collections | Organized, persistent vector storage |
| Automatic embedding | DB handles embedding for you |
| Semantic queries | Search by meaning with distance scores |
| Metadata filtering | Filter results by source, topic, date |
| CRUD operations | Add, update, delete documents |

**Dataset:** The same 2026 Iran War corpus, now stored in a proper vector database.

**Time:** ~25 minutes

## Why Vector Databases?

In Session 2 we built `SimpleSemanticSearch` — it works, but:

| Problem | Vector DB solution |
|---|---|
| All data in RAM — lost on restart | Persistent storage |
| Linear scan over all documents | Fast approximate nearest neighbor (ANN) search |
| No filtering by metadata | Rich metadata queries |
| No updates or deletes | Full CRUD operations |

**ChromaDB** is a lightweight, open-source vector database perfect for learning and prototyping.

## Setup

In [1]:
!pip install --upgrade pip -q
!pip install chromadb openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 13.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.27.1 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.27.1 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-

In [9]:
import chromadb
from openai import OpenAI
from typing import List, Dict
import os

from google.colab import userdata
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

EMBEDDING_MODEL = "openai/text-embedding-3-small"

# ── Custom embedding function for ChromaDB ──
class OpenRouterEmbeddingFunction:
    """Bridges ChromaDB with OpenRouter's embedding API."""

    def __init__(self, api_key: str, model: str = EMBEDDING_MODEL):
        self.client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
        self.model = model

    def __call__(self, input: List[str]) -> List[List[float]]:
        response = self.client.embeddings.create(input=input, model=self.model)
        return [item.embedding for item in response.data]

    def embed_query(self, input: List[str]) -> List[List[float]]:
      return self.__call__(input)

    def name(self) -> str:
        return self.model

embedding_fn = OpenRouterEmbeddingFunction(os.environ["OPENROUTER_API_KEY"])

print(f"ChromaDB version: {chromadb.__version__}")
print(f"✓ Embedding function ready ({EMBEDDING_MODEL})")

ChromaDB version: 1.5.5
✓ Embedding function ready (openai/text-embedding-3-small)


## Load the Document Corpus

In [10]:
# ── Load the corpus ──
# Upload usa-iran-corpus.txt using the Files panel (📁) on the left,
# or run:  from google.colab import files; uploaded = files.upload()

def load_corpus(path="usa-iran-corpus.txt"):
    """Load documents and metadata from the corpus file."""
    with open(path) as f:
        raw = f.read()

    blocks = raw.split("===DOC===\n")[1:]  # skip text before first delimiter
    documents, metadatas = [], []

    for block in blocks:
        header_part, content = block.split("===\n", 1)
        meta = {}
        for line in header_part.strip().split("\n"):
            if ": " in line:
                key, val = line.split(": ", 1)
                meta[key.strip()] = val.strip()
        documents.append(content.strip())
        metadatas.append(meta)

    return documents, metadatas

documents, metadatas = load_corpus()
print(f"✓ Loaded {len(documents)} documents")
for i in range(3):
    title = documents[i].split("\n")[0][:70]
    print(f"  [{i}] ({metadatas[i].get('topic','?')}) {title}")
print(f"  ... and {len(documents) - 3} more")

✓ Loaded 23 documents
  [0] (overview) # The 2026 Iran War: Comprehensive Overview
  [1] (reporting) # US-Israel War on Iran: Day 25 of Attacks
  [2] (diplomatic) # Trump Says U.S. in Talks with Iran to End War; Iran Denies It
  ... and 20 more


## Part 1: Create a Client and Collection

ChromaDB hierarchy: **Client** → **Collections** → **Documents**

A collection is like a table — it groups related documents with a shared embedding function.

In [11]:
# In-memory client (data lives only during this session)
db = chromadb.Client()

# Create a collection — ChromaDB will auto-embed documents using our function
collection = db.get_or_create_collection(
    name="iran_war_2026",
    embedding_function=embedding_fn,
    metadata={"description": "2026 Iran War analysis articles"}
)

print(f"✓ Collection '{collection.name}' ready")

✓ Collection 'iran_war_2026' ready


## Part 2: Adding Documents

Each document in ChromaDB has:
- **id** — unique identifier
- **document** — the text (ChromaDB auto-generates the embedding)
- **metadata** — structured fields for filtering

In [12]:
# Prepare IDs and add documents with their metadata
ids = [f"doc_{i:02d}" for i in range(len(documents))]

collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas,
)

print(f"✓ Added {collection.count()} documents to '{collection.name}'")
print(f"\nSample metadata:")
for i in range(3):
    print(f"  {ids[i]}: {metadatas[i]}")

✓ Added 23 documents to 'iran_war_2026'

Sample metadata:
  doc_00: {'topic': 'overview', 'source': 'Britannica'}
  doc_01: {'topic': 'reporting', 'source': 'Al Jazeera', 'date': 'March 24, 2026'}
  doc_02: {'topic': 'diplomatic', 'source': 'NPR', 'date': 'March 23, 2026'}


In [13]:
# Inspect what's stored
sample = collection.peek(limit=2)

print("Stored data structure:")
print(f"  Keys: {list(sample.keys())}")
print(f"\nFirst document:")
print(f"  ID:       {sample['ids'][0]}")
print(f"  Text:     {sample['documents'][0][:80]}...")
print(f"  Metadata: {sample['metadatas'][0]}")
if sample["embeddings"] is not None and len(sample["embeddings"]) > 0:
    print(f"  Embedding: {len(sample['embeddings'][0])} dimensions")

Stored data structure:
  Keys: ['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas']

First document:
  ID:       doc_00
  Text:     # The 2026 Iran War: Comprehensive Overview

**Source:** Britannica
**URL:** htt...
  Metadata: {'topic': 'overview', 'source': 'Britannica'}
  Embedding: 1536 dimensions


## Part 3: Querying Documents

ChromaDB converts your query to an embedding, finds the nearest vectors, and returns results with **distance** scores.

> **Distance** = 1 − similarity (lower distance = more similar)

In [14]:
query = "What happened to Iran's nuclear program?"

results = collection.query(query_texts=[query], n_results=3)

print(f"Query: '{query}'")
print("=" * 80)

for doc_id, doc, meta, dist in zip(
    results["ids"][0],
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    similarity = 1 - dist
    title = doc.split("\n")[0][:70]
    print(f"\n  [{doc_id}] Similarity: {similarity:.3f} | Topic: {meta['topic']} | Source: {meta['source']}")
    print(f"  {title}")

Query: 'What happened to Iran's nuclear program?'

  [doc_06] Similarity: 0.049 | Topic: military | Source: Center for Strategic and International Studies (CSIS)
  # Operation Epic Fury and the Remnants of Iran's Nuclear Program

  [doc_05] Similarity: 0.047 | Topic: nuclear | Source: Arms Control Association
  # The U.S. War on Iran: New and Lingering Nuclear Risks

  [doc_15] Similarity: -0.007 | Topic: nuclear | Source: Arms Control Association
  # U.S. Negotiators Were Ill-Prepared for Serious Nuclear Negotiations 


In [15]:
query = "How did the war affect the global economy?"

results = collection.query(query_texts=[query], n_results=3)

print(f"Query: '{query}'")
print("=" * 80)

for doc_id, doc, meta, dist in zip(
    results["ids"][0],
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    similarity = 1 - dist
    title = doc.split("\n")[0][:70]
    print(f"\n  [{doc_id}] Similarity: {similarity:.3f} | Topic: {meta['topic']} | Source: {meta['source']}")
    print(f"  {title}")

Query: 'How did the war affect the global economy?'

  [doc_08] Similarity: -0.093 | Topic: economic | Source: Council on Foreign Relations (CFR)
  # The Iran War is Causing Energy Chaos in Asia

  [doc_11] Similarity: -0.108 | Topic: economic | Source: Federal Reserve Bank of Dallas
  # Strait of Hormuz Closure: Global Economic Impact Analysis

  [doc_09] Similarity: -0.120 | Topic: economic | Source: Foreign Policy
  # Iran War's Impact on Gas Markets in Europe, Asia Will Last for Years


## Part 4: Metadata Filtering

ChromaDB can **filter before searching** — combine semantic similarity with structured metadata.

This is powerful: "find relevant documents, but *only* from economic sources."

In [16]:
# Filter by topic
query = "What are the consequences of the conflict?"

results = collection.query(
    query_texts=[query],
    n_results=3,
    where={"topic": "economic"},
)

print(f"Query: '{query}'")
print(f"Filter: topic = 'economic'")
print("=" * 80)

for doc, meta, dist in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    similarity = 1 - dist
    title = doc.split("\n")[0][:70]
    print(f"\n  Similarity: {similarity:.3f} | Source: {meta['source']}")
    print(f"  {title}")

Query: 'What are the consequences of the conflict?'
Filter: topic = 'economic'

  Similarity: -0.208 | Source: Council on Foreign Relations (CFR)
  # The Iran War is Causing Energy Chaos in Asia

  Similarity: -0.241 | Source: Atlantic Council
  # How the Iran War Could Trigger a European Energy Crisis

  Similarity: -0.276 | Source: Foreign Policy
  # Iran War's Impact on Gas Markets in Europe, Asia Will Last for Years


In [17]:
# Compound filter: economic OR humanitarian topics
query = "How are civilians affected?"

results = collection.query(
    query_texts=[query],
    n_results=4,
    where={
        "$or": [
            {"topic": "humanitarian"},
            {"topic": "economic"},
        ]
    },
)

print(f"Query: '{query}'")
print(f"Filter: topic = 'humanitarian' OR 'economic'")
print("=" * 80)

for doc, meta, dist in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    similarity = 1 - dist
    title = doc.split("\n")[0][:70]
    print(f"\n  Similarity: {similarity:.3f} | Topic: {meta['topic']} | Source: {meta['source']}")
    print(f"  {title}")

Query: 'How are civilians affected?'
Filter: topic = 'humanitarian' OR 'economic'

  Similarity: -0.113 | Topic: humanitarian | Source: Refugees International
  # U.S./Israel-Iran War on Course for Cataclysmic Civilian Harm, Displa

  Similarity: -0.354 | Topic: humanitarian | Source: Amnesty International
  # USA/Iran: US Strike on School That Killed Over 100 Children Must Fac

  Similarity: -0.391 | Topic: economic | Source: Council on Foreign Relations (CFR)
  # The Iran War is Causing Energy Chaos in Asia

  Similarity: -0.407 | Topic: economic | Source: Federal Reserve Bank of Dallas
  # Strait of Hormuz Closure: Global Economic Impact Analysis


## Part 5: Update and Delete

ChromaDB supports full CRUD — important for keeping your knowledge base current.

In [18]:
# UPDATE a document's metadata
collection.update(
    ids=["doc_00"],
    metadatas=[{**metadatas[0], "reviewed": True}],
)

updated = collection.get(ids=["doc_00"])
print("Updated metadata:")
print(f"  {updated['metadatas'][0]}")

Updated metadata:
  {'reviewed': True, 'topic': 'overview', 'source': 'Britannica'}


In [19]:
# DELETE by ID
print(f"Before delete: {collection.count()} documents")
collection.delete(ids=["doc_00"])
print(f"After delete:  {collection.count()} documents")

Before delete: 23 documents
After delete:  22 documents


In [20]:
# DELETE by metadata filter
print(f"Before delete: {collection.count()} documents")
collection.delete(where={"topic": "legal"})
print(f"After delete:  {collection.count()} documents (removed 'legal' docs)")

Before delete: 22 documents
After delete:  20 documents (removed 'legal' docs)


### 🔬 Your turn

Try your own queries and filters. Available topics: `overview`, `reporting`, `military`, `humanitarian`, `economic`, `cyber`, `nuclear`, `diplomatic`, `legal`, `analysis`

In [21]:
# ── Try your own query + filter ──
your_query = "What about cyber warfare?"  # ← change this

results = collection.query(
    query_texts=[your_query],
    n_results=3,
    # where={"topic": "cyber"},  # ← uncomment to filter
)

for doc, meta, dist in zip(
    results["documents"][0], results["metadatas"][0], results["distances"][0]
):
    sim = 1 - dist
    title = doc.split("\n")[0][:70]
    print(f"[{sim:.3f}] ({meta['topic']}, {meta['source']}) {title}")

[-0.030] (cyber, PBS NewsHour) # Iran-Linked Hackers Target U.S. and Allies Amid Ongoing Conflict
[-0.060] (cyber, Unit 42 / Palo Alto Networks) # Threat Brief: March 2026 Escalation of Cyber Risk Related to Iran
[-0.289] (analysis, Brookings Institution) # After the Strike: The Danger of War in Iran


## Key Takeaways

| Concept | What you learned |
|---|---|
| **Vector database** | Persistent, efficient semantic storage |
| **Collections** | Group documents + auto-generate embeddings |
| **Querying** | Semantic search with distance/similarity scores |
| **Metadata filtering** | Combine meaning-based search with structured filters |
| **CRUD** | Add, inspect, update, and delete documents |

**What's next (future workshop):** Combine retrieval (ChromaDB) with generation (LLM) → a full **RAG system** that answers questions grounded in these documents.